The fixture below plants the kind of extract that looks fine in a
spreadsheet and fails a join: an internal key that would be clean if the
vendor did not double-send two orders, an external reference that is
*almost* unique, invoice numbers in two formats, one hole in the business
calendar, and two exact duplicate rows.

In [ ]:
import datetime as dt
import random

import polars as pl

import onceover

rng = random.Random(2024)
n = 4_000

order_id = list(range(n))
external_ref = [f"ORD-{i:06}" for i in range(n)]
for k in range(3):                       # three duplicated references
    external_ref[n - 1 - k] = external_ref[k]
region_id = [rng.randrange(3) for _ in range(n)]
region_name = [["emea", "amer", "apac"][r] for r in region_id]
invoice_no = [f"INV-{i:05}" if i % 2 == 0 else f"{i:05}" for i in range(n)]
quantity = [rng.randrange(1, 11) for _ in range(n)]
unit_price = [round(rng.uniform(5, 25), 2) for _ in range(n)]
line_total = [round(q * p, 2) for q, p in zip(quantity, unit_price)]

base = dt.date(2024, 3, 4)
skip = {dt.date(2024, 4, 2), dt.date(2024, 4, 3)}     # two weekdays missing
business, i = [], 0
while len(business) < n:
    d = base + dt.timedelta(days=i // 20)
    if d not in skip:
        business.append(d)
    i += 1

df = pl.DataFrame({
    "order_id": order_id,
    "external_ref": external_ref,
    "region_id": region_id,
    "region_name": region_name,
    "invoice_no": invoice_no,
    "quantity": quantity,
    "unit_price": unit_price,
    "line_total": line_total,
    "business_date": business,
})
df = pl.concat([df, df[500:502]])        # two duplicated rows, on purpose

report = onceover.profile(df)
print(report.overview["headline"])

The headline reports no key candidate. The two repeated orders mean no
single column is unique across all rows. Review the ranked findings to
see which issues affect joins, date checks, and invoice-number parsing.

In [ ]:
for f in report.findings:
    kind = "/".join(p for p in (f["category"], f.get("type")) if p)
    print(f"[{f['interestingness']:.2f} {kind}] {f['claim']}")

**Deduplicate, then trust the key.** `order_id` is flagged nearly
unique — two repeated values — and the duplicates finding explains them:
two rows repeat earlier values across every compared column. Full-row
exact equality, missing values comparing equal, the comparison stated in
the specification. These are re-sends, not collisions: deduplicate on
`order_id`, log the redundant rows, and the key story closes.

**The external reference is not a key.** `external_ref` is nearly unique
with *five* repeated values, and the duplicates finding does not explain
them — the rows behind repeated references differ. That is the
join-breaker: do not join the vendor's other extracts on `external_ref`
until they confirm whether the repeats are double-submissions or a
truncation bug.

In [ ]:
dirty = next(f for f in report.findings
              if f["category"] == "uniqueness" and "external_ref" in f["subject"]["columns"])
for e in dirty["evidence"]:
    if "examples" in e["metrics"]:
        print("repeated external_ref values:", e["metrics"]["examples"])

In [ ]:
decisions = pl.DataFrame({
    "finding": [
        "order_id nearly unique; 2 rows repeat earlier values exactly",
        "external_ref nearly unique (5 repeats, rows differ)",
        "invoice_no mixes 2 formats",
        "2 business dates missing",
        "line_total predictable from quantity × unit_price",
    ],
    "decision": [
        "deduplicate on order_id; then load key",
        "do not join on it; query vendor",
        "normalize in staging before parsing",
        "send dates back to vendor",
        "keep as recomputation check column",
    ],
    "basis": ["measured · full", "measured · full",
              "measured · full", "measured · full", "estimated · holdout"],
})

## What we did *not* conclude

The report found no functional dependency between `quantity` and
`line_total` — correctly, because the mapping runs through `unit_price`.
It did not flag `region_name → region_id` as a data-quality problem —
it is a lookup, and the finding says so with both directions verified and
zero violations. And absence beyond this ledger is not proof: the
coverage block shows which passes ran, sampled, or were capped, and on
4,000 rows everything ran at full scope except the model passes' encoded
sample ([](../usage/coverage.ipynb)).

The pattern generalizes: profile once, read the measured statements
first, convert each into a decision with the evidence class attached,
and route the rest to whoever owns the data.